# Olist E-commerce BI Platform: Data cleaning (Layer 2, v1)

Cleans the 9 raw Olist CSVs and exports a star schema for Power BI:

- **Facts:** `fact_orders` (order grain, delivery + review metrics), `fact_order_items` (sales grain), `fact_reviews`, `fact_payments`
- **Dimensions:** `dim_customer`, `dim_seller`, `dim_product`, `dim_date`

**Before running:** put the 9 Kaggle CSVs in a folder called `olist_raw` next to this notebook.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW = Path("olist_raw")    # folder containing the 9 Kaggle CSVs
OUT = Path("olist_clean")  # cleaned files for Power BI go here
OUT.mkdir(exist_ok=True)

## 1. Load raw files

In [ ]:
files = {
    "orders":      "olist_orders_dataset.csv",
    "items":       "olist_order_items_dataset.csv",
    "payments":    "olist_order_payments_dataset.csv",
    "reviews":     "olist_order_reviews_dataset.csv",
    "customers":   "olist_customers_dataset.csv",
    "sellers":     "olist_sellers_dataset.csv",
    "products":    "olist_products_dataset.csv",
    "geo":         "olist_geolocation_dataset.csv",
    "translation": "product_category_name_translation.csv",
}
df = {name: pd.read_csv(RAW / f) for name, f in files.items()}
for name, d in df.items():
    print(f"{name:12} rows={d.shape[0]:>8,}  cols={d.shape[1]}")

## 2. Orders → `fact_orders`

Key fix: in Olist, `customer_id` is a **new id for every order**. The real person is `customer_unique_id`, so we use that as the customer key. Without this, every customer looks like a first-time buyer.

In [ ]:
orders = df["orders"].copy()

date_cols = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
             "order_delivered_customer_date", "order_estimated_delivery_date"]
for c in date_cols:
    orders[c] = pd.to_datetime(orders[c], errors="coerce")

# Swap per-order customer_id for the real customer key
orders = orders.merge(df["customers"][["customer_id", "customer_unique_id"]], on="customer_id", how="left")

orders["order_date"] = orders["order_purchase_timestamp"].dt.normalize()
orders["is_delivered"] = orders["order_delivered_customer_date"].notna().astype(int)

# Delivery metrics
orders["delivery_days"] = ((orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"])
                           .dt.total_seconds() / 86400).round(1)
orders["delay_days"] = (orders["order_delivered_customer_date"].dt.normalize()
                        - orders["order_estimated_delivery_date"].dt.normalize()).dt.days
orders["is_late"] = (orders["delay_days"] > 0).astype("Int64").where(orders["is_delivered"] == 1)

orders["delivery_bucket"] = (
    pd.cut(orders["delay_days"],
           bins=[-np.inf, -8, -1, 0, 7, np.inf],
           labels=["Early 8+ days", "Early 1-7 days", "On time", "Late 1-7 days", "Late 8+ days"])
    .astype(object)
    .where(orders["is_delivered"] == 1, "Not delivered")
)
orders["delivery_bucket_sort"] = orders["delivery_bucket"].map(
    {"Early 8+ days": 1, "Early 1-7 days": 2, "On time": 3, "Late 1-7 days": 4, "Late 8+ days": 5, "Not delivered": 6})

orders["order_status"] = orders["order_status"].str.title()
orders.drop(columns="customer_id", inplace=True)
orders[["order_status"]].value_counts()

## 3. Reviews → `fact_reviews` (+ latest score onto each order)

In [ ]:
reviews = df["reviews"].copy()
for c in ["review_creation_date", "review_answer_timestamp"]:
    reviews[c] = pd.to_datetime(reviews[c], errors="coerce")

reviews = reviews.drop_duplicates(["review_id", "order_id"])

# Remove line breaks so the text imports cleanly into Power BI
for c in ["review_comment_title", "review_comment_message"]:
    reviews[c] = reviews[c].astype("string").str.replace(r"[\r\n]+", " ", regex=True).str.strip()
reviews["has_comment"] = reviews["review_comment_message"].notna().astype(int)

# One score per order (latest review) so delay vs. score analysis is simple
latest = (reviews.sort_values("review_answer_timestamp")
                 .drop_duplicates("order_id", keep="last")[["order_id", "review_score"]])
orders = orders.merge(latest, on="order_id", how="left")

fact_reviews = reviews
fact_orders = orders
print(f"Orders with a review: {fact_orders['review_score'].notna().mean():.1%}")

## 4. Order items → `fact_order_items` (sales grain)

In [ ]:
items = df["items"].copy()
items["shipping_limit_date"] = pd.to_datetime(items["shipping_limit_date"], errors="coerce")
items = items.merge(orders[["order_id", "customer_unique_id", "order_date", "order_status"]],
                    on="order_id", how="left")
items["item_total"] = items["price"] + items["freight_value"]

fact_order_items = items[["order_id", "order_item_id", "product_id", "seller_id", "customer_unique_id",
                          "order_date", "order_status", "shipping_limit_date",
                          "price", "freight_value", "item_total"]]
fact_order_items.head()

## 5. Payments → `fact_payments`

In [ ]:
fact_payments = df["payments"].copy()
fact_payments["payment_type"] = fact_payments["payment_type"].str.replace("_", " ").str.title()
fact_payments["payment_type"].value_counts()

## 6. Products → `dim_product` (English categories)

In [ ]:
products = df["products"].merge(df["translation"], on="product_category_name", how="left")

# A few categories have no translation: fall back to the Portuguese name, then 'Unknown'
products["category"] = (products["product_category_name_english"]
                        .fillna(products["product_category_name"])
                        .fillna("unknown")
                        .str.replace("_", " ").str.title())

products = products.rename(columns={"product_name_lenght": "product_name_length",
                                    "product_description_lenght": "product_description_length"})
products["product_volume_cm3"] = (products["product_length_cm"] * products["product_height_cm"]
                                  * products["product_width_cm"])

dim_product = products[["product_id", "category", "product_name_length", "product_description_length",
                        "product_photos_qty", "product_weight_g", "product_volume_cm3"]]
dim_product["category"].nunique(), dim_product.shape

## 7. Geolocation (one lat/lng per zip prefix, Brazil only)

In [ ]:
geo = df["geo"]
geo = geo[geo["geolocation_lat"].between(-34, 6) & geo["geolocation_lng"].between(-74, -34)]
geo = (geo.groupby("geolocation_zip_code_prefix", as_index=False)
          .agg(lat=("geolocation_lat", "mean"), lng=("geolocation_lng", "mean")))
geo.shape

## 8. Customers → `dim_customer` (one row per real customer)

In [ ]:
cust = (df["customers"].drop_duplicates("customer_unique_id", keep="last")
          .merge(geo, left_on="customer_zip_code_prefix", right_on="geolocation_zip_code_prefix", how="left"))
cust["customer_city"] = cust["customer_city"].str.title()

# First order date powers new-vs-repeat and cohort analysis later
first = orders.groupby("customer_unique_id", as_index=False).agg(
    first_order_date=("order_date", "min"), total_orders=("order_id", "nunique"))
cust = cust.merge(first, on="customer_unique_id", how="left")
cust["customer_type"] = np.where(cust["total_orders"] > 1, "Repeat", "One-time")

dim_customer = cust[["customer_unique_id", "customer_city", "customer_state", "customer_zip_code_prefix",
                     "lat", "lng", "first_order_date", "total_orders", "customer_type"]]
dim_customer["customer_type"].value_counts(normalize=True)

## 9. Sellers → `dim_seller`

In [ ]:
sell = df["sellers"].merge(geo, left_on="seller_zip_code_prefix", right_on="geolocation_zip_code_prefix", how="left")
sell["seller_city"] = sell["seller_city"].str.title()
dim_seller = sell[["seller_id", "seller_city", "seller_state", "seller_zip_code_prefix", "lat", "lng"]]
dim_seller.shape

## 10. `dim_date`

In [ ]:
# Full calendar years so DAX time intelligence (YTD, SAMEPERIODLASTYEAR) works correctly
start = f"{fact_orders['order_date'].dt.year.min()}-01-01"
end = f"{fact_orders['order_date'].dt.year.max()}-12-31"
dates = pd.date_range(start, end, freq="D")
dim_date = pd.DataFrame({"date": dates})
dim_date["year"] = dim_date["date"].dt.year
dim_date["quarter"] = "Q" + dim_date["date"].dt.quarter.astype(str)
dim_date["month_num"] = dim_date["date"].dt.month
dim_date["month_name"] = dim_date["date"].dt.strftime("%b")
dim_date["year_month"] = dim_date["date"].dt.strftime("%b %Y")
dim_date["year_month_sort"] = dim_date["year"] * 100 + dim_date["month_num"]
dim_date["weekday_num"] = dim_date["date"].dt.weekday + 1
dim_date["weekday"] = dim_date["date"].dt.strftime("%a")
dim_date["is_weekend"] = (dim_date["weekday_num"] >= 6).astype(int)
dim_date.head()

## 11. Quick insight preview (your star story)

In [ ]:
delivered = fact_orders[fact_orders["is_delivered"] == 1]
print(f"Late delivery rate: {delivered['is_late'].mean():.1%}\n")
print("Average review score by delivery bucket:")
print(delivered.groupby(["delivery_bucket_sort", "delivery_bucket"])["review_score"].mean().round(2))

## 12. Export for Power BI

In [ ]:
tables = {
    "fact_orders": fact_orders, "fact_order_items": fact_order_items,
    "fact_reviews": fact_reviews, "fact_payments": fact_payments,
    "dim_customer": dim_customer, "dim_seller": dim_seller,
    "dim_product": dim_product, "dim_date": dim_date,
}
for name, t in tables.items():
    t.to_csv(OUT / f"{name}.csv", index=False, date_format="%Y-%m-%d %H:%M:%S", encoding="utf-8")
    print(f"{name:18} {t.shape[0]:>8,} rows")
print(f"\nSaved to: {OUT.resolve()}")